# 03 — Linkage-quality evaluation and survival analysis (both layers)

Inputs: only the datasets exported by `01_data_engineering_pipeline.ipynb`
(sensitivity variants are re-generated here from those exports with the same
`src/boamp` functions).

**Evidence classification used throughout** — every result is labeled as one of:

| Class | Meaning |
|---|---|
| internal diagnostic | computed from the same scores that selected the links (weakest) |
| indirect validation | independent structure in the data (e.g. corroborating fields) |
| synthetic validation | controlled perturbation experiments (corruption/recovery) |
| manually reviewed | human labels — **none completed yet**, samples remain unlabeled |
| externally verified | outside ground truth — **none exists** for renewals |
| unverified assumption | stated, tested for sensitivity, but not verifiable |

No precision/recall claim below is ground-truth based: the Fellegi–Sunter
estimates are model-based and conditional on blocking.

**Deliberate design note:** evaluation results here do NOT feed back into the
score weights or thresholds (the classical evaluation→comparison loop is left
open) — tuning the comparison on links selected by the same score would be
circular. Sensitivity analyses replace that loop.


In [1]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT / "src"))

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from boamp.config import load_config
from boamp.reporting.figures import setup_style, save_figure

cfg = load_config(PROJECT_ROOT)
setup_style()
P = cfg.pipeline
D1, D2, D3 = cfg.paths.processed_boamp_only, cfg.paths.processed_enriched, cfg.paths.processed_comparison
T = cfg.paths.reports_tables

DATE_COLS = ["publication_date", "start_date", "estimated_end_date", "study_end_date"]
STR_COLS = {c: str for c in ["notice_id", "buyer_key", "buyer_key_type", "buyer_name_normalized",
                             "buyer_siret_clean", "buyer_siren_clean", "objet_clean", "cpv_clean",
                             "cpv_division", "cpv_category", "cpv_class", "cpv_group", "category_label"]}

sources = pd.read_csv(D1 / "boamp_only_sources.csv", dtype=STR_COLS, parse_dates=DATE_COLS)
sources["dur_was_imputed"] = sources["dur_was_imputed"].astype(str).str.lower().eq("true")
sources["cpv_generic_flag"] = sources["cpv_generic_flag"].astype(str).str.lower().eq("true")
l1_pairs = pd.read_csv(D1 / "boamp_only_candidate_pairs.csv",
                       parse_dates=["source_date", "candidate_date", "expected_end_date"])
l2_pairs = pd.read_csv(D2 / "enriched_candidate_pairs.csv")
l1_links = {v: pd.read_csv(D1 / f"boamp_only_links_{v}.csv") for v in ("broad", "balanced", "strict")}
l2_links = {v: pd.read_csv(D2 / f"enriched_links_{v}.csv") for v in ("broad", "balanced", "strict")}
surv_l1 = pd.read_csv(D1 / "boamp_only_survival.csv", parse_dates=DATE_COLS)
surv_l2 = pd.read_csv(D2 / "enriched_survival.csv", parse_dates=DATE_COLS)
link_cmp = pd.read_csv(D3 / "layer_link_comparison.csv")
WINDOW = int((D1 / "window_months.txt").read_text().strip())
N_ELIGIBLE = int((sources["buyer_key_type"] != "MISSING").sum())
THRESHOLDS = {k: getattr(P.thresholds, k) for k in ("broad", "balanced", "strict")}
print(f"eligible {N_ELIGIBLE} | window {WINDOW}m | L1 balanced {len(l1_links['balanced'])} | "
      f"L2 balanced {len(l2_links['balanced'])}")

eligible 3159 | window 6m | L1 balanced 618 | L2 balanced 847


# Part 1 — Linkage-quality evaluation

## 1.1 Fellegi–Sunter Beta-mixture (model-based; *internal diagnostic*)

Two-class mixture over all blocked candidate pairs, per layer. `precision_hat`
and `recall_hat` are functions of the fitted posterior — **not** ground truth —
and recall is defined **only over pairs that survived blocking** (the 1,923
zero-candidate sources are structurally outside it).

In [2]:
from boamp.validation.linkage_quality import fit_fs_beta_mixture

fs = {}
for layer, pairs, links in [("boamp_only", l1_pairs, l1_links["balanced"]),
                            ("enriched", l2_pairs, l2_links["balanced"])]:
    print(f"--- {layer} ---")
    fs[layer] = fit_fs_beta_mixture(pairs, links)
    fs[layer]["precision_recall"].insert(0, "layer", layer)
    fs[layer]["params"].insert(0, "layer", layer)
fs_pr = pd.concat([fs[k]["precision_recall"] for k in fs], ignore_index=True)
fs_pr.to_csv(T / "linkage_fs_precision_recall_by_layer.csv", index=False)
pd.concat([fs[k]["params"] for k in fs], ignore_index=True).to_csv(T / "linkage_fs_params_by_layer.csv", index=False)
display(fs_pr[["layer", "precision_hat", "recall_hat", "n_linked", "n_omega"]])
for k in fs:
    display(fs[k]["conditional_independence"].assign(layer=k))

--- boamp_only ---


best run warm_start logL=40975.5; p_hat=0.1809; precision_hat=0.4552, recall_hat=0.2534 (recall over blocked pairs only)
--- enriched ---


best run warm_start logL=36294.9; p_hat=0.6808; precision_hat=0.7386, recall_hat=0.1117 (recall over blocked pairs only)


,layer,precision_hat,recall_hat,n_linked,n_omega
0,boamp_only,0.455236,0.253361,618,6137
1,enriched,0.738635,0.111681,847,8228


,n_linked_pairs,pearson_r_cpv_text,pearson_p_cpv_text,spearman_r_cpv_text,spearman_p_cpv_text,flag_high_correlation_gt_0_3,layer
0,618,0.046535,0.248035,-0.105593,0.008614,False,boamp_only


,n_linked_pairs,pearson_r_cpv_text,pearson_p_cpv_text,spearman_r_cpv_text,spearman_p_cpv_text,flag_high_correlation_gt_0_3,layer
0,847,0.097846,0.004368,-0.045894,0.182073,False,enriched


## 1.2 Corruption / recovery (*synthetic validation*)

Strict-tier Layer 1 links are rescored after controlled corruption of text, CPV,
and duration. `delta_R = R(0) - R(3)` ranks which field the linkage actually
depends on. The R(0)=1 sanity gate proves the rescoring reproduces the real
pipeline.

In [3]:
from boamp.validation.corruption import run_corruption_recovery

m4_scores, m4_by_sev, m4_summary = run_corruption_recovery(
    sources, l1_pairs, l1_links["strict"], THRESHOLDS["balanced"], WINDOW, cfg)
m4_by_sev.to_csv(T / "linkage_corruption_recovery_by_severity.csv", index=False)
m4_summary.to_csv(T / "linkage_corruption_recovery_summary.csv", index=False)
display(m4_summary)
fig, ax = plt.subplots(figsize=(7, 4))
for sweep, grp in m4_by_sev.groupby("sweep"):
    ax.plot(grp["severity"], grp["R"], marker="o", label=sweep)
ax.set_xlabel("corruption severity"); ax.set_ylabel("recovery rate R")
ax.set_xticks([0, 1, 2, 3]); ax.legend(); ax.set_title("Corruption-recovery curves (L1 strict links)")
save_figure(fig, "an_corruption_recovery", cfg)
plt.show()

Sanity gate passed: R(0) == 1.0 for every sweep.


,sweep,R0,R1,R2,R3,delta_R
3,combined,1.0,0.906149,0.653722,0.524272,0.475728
0,text_only,1.0,0.915858,0.776699,0.718447,0.281553
1,cpv_only,1.0,0.990291,0.983819,0.851133,0.148867
2,duration_only,1.0,0.987055,0.864078,0.980583,0.019417


## 1.3 Threshold sensitivity (*internal diagnostic*)

How many rank-1 sources sit within ±δ of the balanced threshold — the flip risk
of small cutoff changes.

In [4]:
from boamp.validation.linkage_quality import threshold_sweep, local_threshold_sensitivity

sweeps = []
for layer, pairs in [("boamp_only", l1_pairs), ("enriched", l2_pairs)]:
    sw = threshold_sweep(pairs, N_ELIGIBLE)
    sw.insert(0, "layer", layer)
    sweeps.append(sw)
sweep_df = pd.concat(sweeps, ignore_index=True)
sweep_df.to_csv(T / "linkage_threshold_sweep_by_layer.csv", index=False)
sens = local_threshold_sensitivity(l1_pairs, THRESHOLDS)
sens.to_csv(T / "linkage_threshold_local_sensitivity.csv", index=False)
display(sens[["delta", "n_flipped", "pct_flipped"]])
fig, ax = plt.subplots(figsize=(7, 4))
for layer, grp in sweep_df.groupby("layer"):
    ax.plot(grp["percentile"], grp["event_rate"], marker="o", label=layer)
for pct in (25, 50, 75):
    ax.axvline(pct, ls=":", color="gray", alpha=0.6)
ax.set_xlabel("threshold percentile of rank-1 scores"); ax.set_ylabel("linking rate")
ax.legend(); ax.set_title("Linking rate vs threshold percentile")
save_figure(fig, "an_threshold_sweep", cfg)
plt.show()

,delta,n_flipped,pct_flipped
0,0.005,48,0.038835
1,0.010,88,0.071197
2,0.020,227,0.183657
3,0.050,516,0.417476


## 1.4 Weight ablation (*internal diagnostic*)

Drop each component, renormalize, re-rank, re-threshold at the ablated rank-1
median; Jaccard vs the real balanced set shows which component drives selection.
The weights are fixed a priori — this quantifies (does not justify) them.

In [5]:
from boamp.validation.linkage_quality import weight_ablation

abl = weight_ablation(l1_pairs, l1_links["balanced"], cfg)
abl.to_csv(T / "linkage_weight_ablation.csv", index=False)
display(abl[["dropped_component", "n_linked_ablated", "jaccard_vs_real_balanced"]])

,dropped_component,n_linked_ablated,jaccard_vs_real_balanced
2,time,618,0.668016
0,text,618,0.683924
1,cpv,618,0.750708
3,buyer,618,0.904468


## 1.5 Temporal-window sensitivity (*internal diagnostic / robustness*)

Candidate generation re-run at 6/9/12/18-month windows (same code path, forced
window). Variant link/survival datasets are exported for the survival section.

In [6]:
from boamp.linkage.candidates import generate_pairs_single_key
from boamp.linkage.links import build_links
from boamp.survival.datasets import build_survival_dataset
from boamp.linkage.scoring import derive_thresholds

ref_keys = set(zip(l1_links["balanced"]["source_notice_id"], l1_links["balanced"]["candidate_notice_id"]))
win_rows = []
for W in P.temporal_window.sensitivity_windows_months:
    pw, _ = generate_pairs_single_key(sources, cfg, verbose=False, window_override=W)
    thr_w = derive_thresholds(pw, cfg)["balanced"]  # p50 of that window's rank-1 scores
    lw = build_links(pw, thr_w, f"window_{W}m", cfg)
    sv = build_survival_dataset(sources, lw, f"window_{W}m", cfg)
    lw.to_csv(D1 / f"boamp_only_links_window_{W}m.csv", index=False)
    sv.to_csv(D1 / f"boamp_only_survival_window_{W}m.csv", index=False)
    keys = set(zip(lw["source_notice_id"], lw["candidate_notice_id"]))
    win_rows.append(dict(window_months=W, n_pairs=len(pw), n_links=len(lw),
                         linking_rate=len(lw) / N_ELIGIBLE, threshold_p50=thr_w,
                         jaccard_vs_reference=len(keys & ref_keys) / len(keys | ref_keys)))
    print(f"window {W}m: pairs={len(pw)}, links={len(lw)}")
win_df = pd.DataFrame(win_rows)
win_df.to_csv(T / "linkage_window_sensitivity.csv", index=False)
display(win_df)

window 6m: pairs=6137, links=618


window 9m: pairs=8692, links=693


window 12m: pairs=10793, links=754


window 18m: pairs=14097, links=834


,window_months,n_pairs,n_links,linking_rate,threshold_p50,jaccard_vs_reference
0,6,6137,618,0.195632,0.323022,1.000000
1,9,8692,693,0.219373,0.334401,0.736424
2,12,10793,754,0.238683,0.344200,0.578826
3,18,14097,834,0.264008,0.354294,0.453453


## 1.6 Duration-leakage check (*internal diagnostic — high priority*)

88% of durations are imputed and the duration enters both blocking (expected end)
and the temporal score. Two counterfactual link sets test whether it drives the
survival conclusions:
- **no-temporal rescoring** of the same candidate pool;
- **forward-24m** generation with no duration/expected-end input at all.

In [7]:
from boamp.linkage.scoring import build_tfidf_matrix
from boamp.validation.linkage_quality import forward_pairs, make_links_from_pairs, no_temporal_weights

nt = l1_pairs.copy()
w_nt = no_temporal_weights(cfg)
nt["score_no_temporal"] = (w_nt["s_text"] * nt["s_text"] + w_nt["s_cpv"] * nt["s_cpv"]
                           + w_nt["s_buyer"] * nt["s_buyer"])
nt_links, nt_thr = make_links_from_pairs(nt, "no_temporal_score_reference_pool", "score_no_temporal")
sv_nt = build_survival_dataset(sources, nt_links, "no_temporal_score_reference_pool", cfg)
sv_nt.to_csv(D1 / "boamp_only_survival_no_temporal.csv", index=False)

eligible = sources[sources["buyer_key_type"] != "MISSING"].sort_values(
    ["buyer_key", "publication_date"]).reset_index(drop=True)
_, tfidf = build_tfidf_matrix(eligible["objet_clean"].fillna("").tolist(), cfg)
fp = forward_pairs(eligible, tfidf, cfg, months=P.evaluation.duration_leakage_forward_months)
f_links, f_thr = make_links_from_pairs(fp, "forward_24m_no_duration", "score_forward_no_duration")
sv_fw = build_survival_dataset(sources, f_links, "forward_24m_no_duration", cfg)
sv_fw.to_csv(D1 / "boamp_only_survival_forward_24m.csv", index=False)

nt_keys = set(zip(nt_links["source_notice_id"], nt_links["candidate_notice_id"]))
f_keys = set(zip(f_links["source_notice_id"], f_links["candidate_notice_id"]))
leak = pd.DataFrame([
    dict(variant="no_temporal_rescoring", n_links=len(nt_links),
         jaccard_vs_balanced=len(nt_keys & ref_keys) / len(nt_keys | ref_keys)),
    dict(variant="forward_24m_no_duration", n_links=len(f_links),
         jaccard_vs_balanced=len(f_keys & ref_keys) / len(f_keys | ref_keys)),
])
leak.to_csv(T / "linkage_duration_leakage_variants.csv", index=False)
display(leak)

,variant,n_links,jaccard_vs_balanced
0,no_temporal_rescoring,618,0.414188
1,forward_24m_no_duration,1105,0.064237


## 1.7 Enrichment-specific quality (*internal + indirect*)

The links only enrichment adds are the population that decides its value. Their
markedly weaker text similarity is the main quality warning; the confidence-tier
and mechanism breakdowns show where the risk concentrates.

In [8]:
added_ids = set(link_cmp.loc[link_cmp["link_status"] == "ADDED_BY_ENRICHMENT", "source_notice_id"])
added = l2_links["balanced"][l2_links["balanced"]["source_notice_id"].isin(added_ids)]
common_l2 = l2_links["balanced"][~l2_links["balanced"]["source_notice_id"].isin(added_ids)]
qual = pd.DataFrame({
    "links_added_by_enrichment": added[["s_text", "s_cpv", "composite_score", "top1_top2_margin"]].median(),
    "links_shared_with_l1": common_l2[["s_text", "s_cpv", "composite_score", "top1_top2_margin"]].median(),
}).T
display(qual)
print("added links by mechanism:")
print(added["buyer_match_mechanism"].value_counts())
print("\nconfidence tiers, L2 balanced:")
print(l2_links["balanced"]["confidence_tier"].value_counts())
print(f"\ncross-establishment (same SIREN, different SIRET) links: "
      f"{added['cross_establishment_same_siren'].astype(bool).sum()} among added, "
      f"{l2_links['balanced']['cross_establishment_same_siren'].astype(bool).sum()} total")
qual.to_csv(T / "enrichment_added_links_quality.csv")

,s_text,s_cpv,composite_score,top1_top2_margin
links_added_by_enrichment,0.069487,0.1,0.359441,0.079995
links_shared_with_l1,0.178842,0.1,0.413130,0.072364


added links by mechanism:
buyer_match_mechanism
HISTORICAL_ALIAS_RECONCILIATION    170
SAME_SIREN_RECONCILIATION           60
Name: count, dtype: int64

confidence tiers, L2 balanced:
confidence_tier
MEDIUM       360
POTENTIAL    328
HIGH         159
Name: count, dtype: int64

cross-establishment (same SIREN, different SIRET) links: 2 among added, 2 total


## 1.8 Evidence-status summary

| Result | Evidence class | Status |
|---|---|---|
| FS precision̂/recall̂ | internal (model-based) | reported with caveats |
| corruption recovery | synthetic | reported |
| threshold/window/weight sensitivity | internal robustness | reported |
| duration leakage | internal robustness | reported — see §2.5 HR reversal |
| enrichment agreement (native vs external SIREN) | indirect | reported in NB01 Part E |
| manual review samples | manually reviewed | **0 labels completed — still open** |
| legal renewal ground truth | external | **does not exist in BOAMP** |


# Part 2 — Survival analysis, Layer 1 vs Layer 2

The substantive question: **does buyer-identity enrichment change the survival
conclusions**, or only the sample composition?

## 2.1 Kaplan–Meier, both layers, + log-rank

In [9]:
from boamp.survival.models import fit_km, km_summary, logrank_between, logrank_by_cpv_division

km_rows = [km_summary(surv_l1, "boamp_only_balanced", cfg),
           km_summary(surv_l2, "enriched_balanced", cfg)]
km_between = logrank_between(surv_l1, surv_l2, "boamp_only", "enriched")
print("log-rank L1 vs L2:", km_between)

fig, ax = plt.subplots(figsize=(8, 5))
for label, sv in [("Layer 1 (boamp_only)", surv_l1), ("Layer 2 (enriched)", surv_l2)]:
    km = fit_km(sv, label)
    km.plot_survival_function(ax=ax, ci_show=True)
ax.set_xlabel("months since source publication"); ax.set_ylabel("S(t) — no renewal yet")
ax.set_title("Kaplan–Meier: time to proxy renewal, both layers")
save_figure(fig, "an_km_by_layer", cfg)
plt.show()
pd.DataFrame(km_rows)

log-rank L1 vs L2: {'group_a': 'boamp_only', 'group_b': 'enriched', 'test_statistic': np.float64(45.3390531004234), 'p': np.float64(1.6571164106296286e-11)}


,variant,n,events,censoring_rate,median_survival_months,survival_at_12m,survival_at_24m,rmst_60m
0,boamp_only_balanced,3159,618,0.804368,inf,0.924267,0.914059,53.513007
1,enriched_balanced,3159,847,0.731877,inf,0.912878,0.897186,51.698705


In [10]:
# stratified KM + pairwise log-rank across digital CPV divisions (both layers)
logrank_rows = [dict(km_between, variant="layer1_vs_layer2")]
for variant, sv in [("boamp_only_balanced", surv_l1), ("enriched_balanced", surv_l2)]:
    logrank_rows += logrank_by_cpv_division(sv, cfg, variant)
logrank_df = pd.DataFrame(logrank_rows)
logrank_df.to_csv(T / "survival_logrank_tests.csv", index=False)  # non-empty now (legacy bug fixed)
display(logrank_df)

,group_a,group_b,test_statistic,p,variant
0,boamp_only,enriched,45.339053,1.657116e-11,layer1_vs_layer2
1,32,35,0.049410,8.240928e-01,boamp_only_balanced
2,32,48,1.547312,2.135327e-01,boamp_only_balanced
3,32,72,15.724210,7.328021e-05,boamp_only_balanced
4,35,48,1.547337,2.135290e-01,boamp_only_balanced
5,35,72,10.446822,1.228607e-03,boamp_only_balanced
6,48,72,10.472017,1.211963e-03,boamp_only_balanced
7,32,35,0.050553,8.221044e-01,enriched_balanced
8,32,48,0.812603,3.673516e-01,enriched_balanced
9,32,72,17.856229,2.382382e-05,enriched_balanced


In [11]:
from boamp.survival.models import normalize_cpv_division
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5), sharey=True)
for ax, (label, sv) in zip(axes, [("Layer 1", surv_l1), ("Layer 2", surv_l2)]):
    svv = sv.copy(); svv["cpv_division"] = normalize_cpv_division(svv["cpv_division"])
    for div in P.scope.digital_cpv_divisions:
        g = svv[svv["cpv_division"] == div]
        if len(g) >= P.survival.km_strata_min_group_size and g["event"].sum() > 0:
            fit_km(g, f"CPV {div} (n={len(g)})").plot_survival_function(ax=ax, ci_show=False)
    ax.set_title(f"{label}: KM by CPV division"); ax.set_xlabel("months")
save_figure(fig, "an_km_by_cpv_division", cfg)
plt.show()

## 2.2 Cox proportional hazards (clustered robust SEs, penalizer 0.01)

Covariates: `log1p(duration)`, imputation flag, CPV-division dummies,
buyer-key-type dummies. Layer 2 clusters on the **enriched** buyer key.
PH tested per term; the quadratic-duration refit checks functional form.

In [12]:
from boamp.survival.models import fit_cox

surv_l2_cox = surv_l2.copy()
surv_l2_cox["buyer_key"] = surv_l2_cox["buyer_key_l2"].fillna(surv_l2_cox["buyer_key"])

cox_rows, ph_frames = [], []
cox_fits = {}
for variant, sv in [("boamp_only_balanced", surv_l1), ("enriched_balanced", surv_l2_cox)]:
    rows, c, x, phdf = fit_cox(sv, variant, cfg)
    cox_rows += rows
    cox_fits[variant] = (c, x, sv)
    if phdf is not None:
        ph_frames.append(phdf)
    qrows, _, _, _ = fit_cox(sv, variant, cfg, quadratic=True)
    cox_rows += [r for r in qrows if r.get("term") in ("log_duration", "duration_sq", "ERROR")]
cox_df = pd.DataFrame(cox_rows)
cox_df.to_csv(T / "survival_cox_by_layer.csv", index=False)
pd.concat(ph_frames, ignore_index=True).to_csv(T / "survival_ph_diagnostics_by_layer.csv", index=False)
display(cox_df[(cox_df["model"] == "cox_clustered")][["variant", "term", "exp_coef", "ci_lower", "ci_upper", "p"]]
        .sort_values(["variant", "p"]).head(20))

/home/senghakrou/survival-analysis/.venv/lib/python3.12/site-packages/lifelines/utils/__init__.py:1120: ConvergenceWarning: Column cpv_division_22 have very low variance when conditioned on death event present or not. This may harm convergence. This could be a form of 'complete separation'. For example, try the following code:

>>> events = df['event'].astype(bool)
>>> print(df.loc[events, 'cpv_division_22'].var())
>>> print(df.loc[~events, 'cpv_division_22'].var())

A very low variance means that the column cpv_division_22 completely determines whether a subject dies or not. See https://stats.stackexchange.com/questions/11109/how-to-deal-with-perfect-separation-in-logistic-regression.

  warnings.warn(dedent(warning_text), ConvergenceWarning)
/home/senghakrou/survival-analysis/.venv/lib/python3.12/site-packages/lifelines/fitters/coxph_fitter.py:1614: ConvergenceWarning: Newton-Raphson failed to converge sufficiently. Please see the following tips in the lifelines documentation: https:

,variant,term,exp_coef,ci_lower,ci_upper,p
12,boamp_only_balanced,cpv_division_43,0.000000,0.000000,0.000000,1.206224e-23
32,boamp_only_balanced,cpv_division_MISSING,3.180070,2.467029,4.099201,4.220523e-19
9,boamp_only_balanced,cpv_division_38,0.166044,0.098017,0.281282,2.450268e-11
6,boamp_only_balanced,cpv_division_33,21.272865,7.786037,58.121325,2.490945e-09
23,boamp_only_balanced,cpv_division_72,1.748783,1.448397,2.111468,6.150125e-09
28,boamp_only_balanced,cpv_division_85,0.165099,0.079176,0.344267,1.555494e-06
1,boamp_only_balanced,dur_was_imputed,0.433097,0.279979,0.669953,1.702220e-04
2,boamp_only_balanced,cpv_division_22,0.195123,0.079543,0.478646,3.579609e-04
7,boamp_only_balanced,cpv_division_34,0.179999,0.065284,0.496287,9.200933e-04
10,boamp_only_balanced,cpv_division_39,6.773467,1.808003,25.375985,4.528595e-03


In [13]:
main_terms = cox_df[(cox_df["model"] == "cox_clustered") & (cox_df["term"] == "log_duration")]
fig, ax = plt.subplots(figsize=(7, 3))
for i, (_, r) in enumerate(main_terms.iterrows()):
    ax.errorbar(r["exp_coef"], i, xerr=[[r["exp_coef"] - r["ci_lower"]], [r["ci_upper"] - r["exp_coef"]]],
                fmt="o", capsize=4)
ax.set_yticks(range(len(main_terms))); ax.set_yticklabels(main_terms["variant"])
ax.axvline(1.0, ls="--", color="gray")
ax.set_xlabel("HR for log_duration (95% CI)"); ax.set_title("Duration hazard ratio by layer")
save_figure(fig, "an_cox_duration_hr", cfg)
plt.show()

## 2.3 Parametric AFT models (Weibull vs log-normal)

In [14]:
from boamp.survival.models import fit_aft

aft_rows = []
for variant, sv in [("boamp_only_balanced", surv_l1), ("enriched_balanced", surv_l2_cox)]:
    aft_rows += fit_aft(sv, variant, cfg)
aft_df = pd.DataFrame(aft_rows)
aft_df.to_csv(T / "survival_aft_by_layer.csv", index=False)
display(aft_df)

,variant,model,AIC,log_likelihood,concordance_index
0,boamp_only_balanced,weibull_aft,8059.765652,-3990.882826,0.713503
1,boamp_only_balanced,lognormal_aft,7948.958002,-3935.479001,0.724833
2,enriched_balanced,weibull_aft,10615.960430,-5268.980215,0.664036
3,enriched_balanced,lognormal_aft,10487.695337,-5204.847668,0.677405


## 2.4 Predictions at 12/24 months + temporal validation

In [15]:
from boamp.survival.models import prediction_calibration, temporal_validation

pred_rows, tv_rows = [], []
for variant, (c, x, sv) in cox_fits.items():
    pred_rows += prediction_calibration(sv, variant, c, x, cfg)
    tv_rows.append(temporal_validation(sv, variant, cfg))
pd.DataFrame(pred_rows).to_csv(T / "survival_predictions_12_24m_by_layer.csv", index=False)
tv_df = pd.DataFrame(tv_rows)
tv_df.to_csv(T / "survival_temporal_validation_by_layer.csv", index=False)
display(tv_df)
pred_df = pd.DataFrame(pred_rows)
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
for ax, h in zip(axes, P.survival.prediction_horizons_months):
    for variant, grp in pred_df[pred_df["horizon_months"] == h].groupby("variant"):
        ax.plot(grp["mean_predicted_risk"], grp["observed_event_rate"], marker="o", label=variant)
    lim = max(pred_df[pred_df["horizon_months"] == h][["mean_predicted_risk", "observed_event_rate"]].max())
    ax.plot([0, lim], [0, lim], ls="--", color="gray")
    ax.set_title(f"calibration at {h} months"); ax.set_xlabel("mean predicted risk"); ax.set_ylabel("observed rate")
    ax.legend(fontsize=8)
save_figure(fig, "an_calibration", cfg)
plt.show()

/home/senghakrou/survival-analysis/.venv/lib/python3.12/site-packages/lifelines/fitters/coxph_fitter.py:1530: LinAlgWarning: An ill-conditioned matrix detected: slice 0 has rcond = 0.0.
  inv_h_dot_g_T = spsolve(-h, g, assume_a="pos", check_finite=False)
/home/senghakrou/survival-analysis/.venv/lib/python3.12/site-packages/lifelines/fitters/coxph_fitter.py:1530: LinAlgWarning: An ill-conditioned matrix detected: slice 0 has rcond = 0.0.
  inv_h_dot_g_T = spsolve(-h, g, assume_a="pos", check_finite=False)


,variant,error
0,boamp_only_balanced,train fit failed or empty test set
1,enriched_balanced,train fit failed or empty test set


## 2.5 Robustness across linkage variants

KM summaries for every variant (incl. `window_6m`, previously generated but never
modeled). The forward-24m variant is where the duration-leakage warning bites:
its `log_duration` HR reverses direction vs the balanced specification — the
duration effect estimated from the balanced links is **not robust** to how the
temporal information enters the linkage, and must not be causally interpreted.

In [16]:
def _load(pth):
    return pd.read_csv(pth, parse_dates=DATE_COLS)

km_var_rows = [km_summary(surv_l1, "boamp_only_balanced", cfg),
               km_summary(surv_l2, "enriched_balanced", cfg)]
for v in ("broad", "strict"):
    lk = l1_links[v]
    sv = build_survival_dataset(sources, lk, v, cfg)
    sv.to_csv(D1 / f"boamp_only_survival_{v}.csv", index=False)
    km_var_rows.append(km_summary(sv, f"boamp_only_{v}", cfg))
for W in P.temporal_window.sensitivity_windows_months:
    sv = _load(D1 / f"boamp_only_survival_window_{W}m.csv")
    km_var_rows.append(km_summary(sv, f"window_{W}m", cfg))
km_var_rows.append(km_summary(_load(D1 / "boamp_only_survival_no_temporal.csv"), "no_temporal_rescoring", cfg))
km_var_rows.append(km_summary(_load(D1 / "boamp_only_survival_forward_24m.csv"), "forward_24m_no_duration", cfg))
km_var = pd.DataFrame(km_var_rows)
km_var.to_csv(T / "survival_km_summary_all_variants.csv", index=False)
display(km_var)

,variant,n,events,censoring_rate,median_survival_months,survival_at_12m,survival_at_24m,rmst_60m
0,boamp_only_balanced,3159,618,0.804368,inf,0.924267,0.914059,53.513007
1,enriched_balanced,3159,847,0.731877,inf,0.912878,0.897186,51.698705
2,boamp_only_broad,3159,927,0.706553,inf,0.900530,0.879792,50.466867
3,boamp_only_strict,3159,309,0.902184,inf,0.963603,0.960146,56.879443
4,window_6m,3159,618,0.804368,inf,0.924267,0.914059,53.513007
5,window_9m,3159,693,0.780627,inf,0.919154,0.904885,52.801371
6,window_12m,3159,754,0.761317,inf,0.916362,0.899389,52.222053
7,window_18m,3159,834,0.735992,inf,0.910278,0.889917,51.361218
8,no_temporal_rescoring,3159,618,0.804368,inf,0.929945,0.920723,53.767652
9,forward_24m_no_duration,3159,1105,0.650206,inf,0.736849,0.636636,41.069625


In [17]:
# duration-leakage Cox check: HR of log_duration under the counterfactual link sets
from lifelines import CoxPHFitter
leak_cox = []
for name, path in [("balanced_reference", None),
                   ("no_temporal_rescoring", D1 / "boamp_only_survival_no_temporal.csv"),
                   ("forward_24m_no_duration", D1 / "boamp_only_survival_forward_24m.csv")]:
    sv = surv_l1 if path is None else _load(path)
    df = sv[["time_to_event_or_censor_months", "event", "declared_duration_months",
             "dur_was_imputed", "buyer_key"]].dropna().copy()
    df = df[df["time_to_event_or_censor_months"] > 0]
    df["log_duration"] = np.log1p(df["declared_duration_months"].astype(float))
    df["dur_was_imputed"] = df["dur_was_imputed"].astype(str).str.lower().eq("true").astype(int)
    c = CoxPHFitter()
    c.fit(df[["time_to_event_or_censor_months", "event", "log_duration", "dur_was_imputed", "buyer_key"]],
          duration_col="time_to_event_or_censor_months", event_col="event",
          cluster_col="buyer_key", robust=True)
    r = c.summary.loc["log_duration"]
    leak_cox.append(dict(variant=name, term="log_duration", exp_coef=r["exp(coef)"],
                         ci_lower=r["exp(coef) lower 95%"], ci_upper=r["exp(coef) upper 95%"], p=r["p"]))
leak_cox_df = pd.DataFrame(leak_cox)
leak_cox_df.to_csv(T / "survival_duration_leakage_cox.csv", index=False)
display(leak_cox_df)
hr_bal = leak_cox_df.loc[leak_cox_df['variant'] == 'balanced_reference', 'exp_coef'].iloc[0]
hr_fwd = leak_cox_df.loc[leak_cox_df['variant'] == 'forward_24m_no_duration', 'exp_coef'].iloc[0]
if (hr_bal - 1) * (hr_fwd - 1) < 0:
    print(f"WARNING (headline finding): log_duration HR REVERSES across specifications "
          f"({hr_bal:.3f} balanced vs {hr_fwd:.3f} forward-24m). The duration covariate "
          f"partly reflects the linkage construction, not contract behavior — do not "
          f"interpret it causally.")

,variant,term,exp_coef,ci_lower,ci_upper,p
0,balanced_reference,log_duration,0.619552,0.542090,0.708082,2.133388e-12
1,no_temporal_rescoring,log_duration,0.697834,0.588029,0.828142,3.809481e-05
2,forward_24m_no_duration,log_duration,1.343931,1.205881,1.497786,9.029596e-08


WARNING (headline finding): log_duration HR REVERSES across specifications (0.620 balanced vs 1.344 forward-24m). The duration covariate partly reflects the linkage construction, not contract behavior — do not interpret it causally.


## 2.6 Do the layers' conclusions differ?

Cross-layer comparison of the substantive quantities. The enriched layer adds
events (mostly via historical aliases), but its added links have weaker text
evidence — so agreement between layers on the survival shape is reassuring, and
disagreement would flag enrichment-induced distortion rather than discovery.

In [18]:
summary_rows = []
for layer, sv in [("boamp_only", surv_l1), ("enriched", surv_l2)]:
    ks = km_summary(sv, layer, cfg)
    cox_hr = cox_df[(cox_df["variant"] == f"{layer}_balanced") & (cox_df["model"] == "cox_clustered")
                    & (cox_df["term"] == "log_duration")]
    aft_ln = aft_df[(aft_df["variant"] == f"{layer}_balanced") & (aft_df["model"] == "lognormal_aft")]
    summary_rows.append({
        "layer": layer,
        "n": ks["n"], "events": ks["events"], "censoring_rate": round(ks["censoring_rate"], 4),
        "survival_at_24m": round(ks["survival_at_24m"], 4),
        f"rmst_{P.survival.rmst_horizon_months}m": round(ks[f"rmst_{P.survival.rmst_horizon_months}m"], 2),
        "median_survival": ks["median_survival_months"],
        "cox_log_duration_HR": round(float(cox_hr["exp_coef"].iloc[0]), 4) if len(cox_hr) else None,
        "lognormal_aft_c_index": round(float(aft_ln["concordance_index"].iloc[0]), 4) if len(aft_ln) else None,
    })
layer_surv_cmp = pd.DataFrame(summary_rows)
layer_surv_cmp.to_csv(T / "survival_layer_conclusions_comparison.csv", index=False)
display(layer_surv_cmp)
print("log-rank L1 vs L2 p-value:", round(km_between["p"], 4))

,layer,n,events,censoring_rate,survival_at_24m,rmst_60m,median_survival,cox_log_duration_HR,lognormal_aft_c_index
0,boamp_only,3159,618,0.8044,0.9141,53.51,inf,0.9231,0.7248
1,enriched,3159,847,0.7319,0.8972,51.70,inf,0.9946,0.6774


log-rank L1 vs L2 p-value: 0.0


In [19]:
print("Analysis complete. Tables in", T, "| figures in", cfg.paths.reports_figures)

Analysis complete. Tables in /home/senghakrou/survival-analysis/reports/tables | figures in /home/senghakrou/survival-analysis/reports/figures
